# P5.1b smoke — TPC-H lineitem at SF=25 (150M rows)

Real data: TPC-H lineitem table at scale factor 25, **149,996,355 rows**.
Streams the (shipdate, disc_price) projection through the chunked Arrow
IPC protocol in `render_mode='density-only'`.

The reference parquet lives in the research Stipple repo at
`Projects/Research/Stipple/app/dist/data/lineitem-sf25.parquet`.

In [1]:
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
from stipple import Stipple
import stipple
print(f'stipple {stipple.__version__}')
PQ_PATH = Path.home() / 'Sites/Thinking/Projects/Research/Stipple/app/dist/data/lineitem-sf25.parquet'
print(f'reading {PQ_PATH}')

stipple 0.1.0
reading /Users/stevo/Sites/Thinking/Projects/Research/Stipple/app/dist/data/lineitem-sf25.parquet


In [2]:
tbl = pq.read_table(PQ_PATH, columns=['l_shipdate', 'l_extendedprice', 'l_discount'])
print(f'rows: {tbl.num_rows:,}')
# x = shipdate as days-since-1992-01-01 ordinal (date32[day] is already int32 days since 1970-01-01)
shipdate_days = tbl.column('l_shipdate').to_numpy(zero_copy_only=False).astype(np.int32)
xs = (shipdate_days - shipdate_days.min()).astype(np.float32)
price = tbl.column('l_extendedprice').to_numpy(zero_copy_only=False).astype(np.float32)
discount = tbl.column('l_discount').to_numpy(zero_copy_only=False).astype(np.float32)
ys = (price * (1.0 - discount)).astype(np.float32)
print(f'x: shipdate-days [{xs.min():.0f}, {xs.max():.0f}]')
print(f'y: disc_price   [{ys.min():.2f}, {ys.max():.2f}]')
del tbl, shipdate_days, price, discount

rows: 149,996,355
x: shipdate-days [0, 2525]
y: disc_price   [810.69, 104948.00]


In [3]:
w = Stipple(x=xs, y=ys, render_mode='density-only')
w

In [4]:
import time
for _ in range(600):
    if w.last_fps:
        break
    time.sleep(0.2)
print(f'status      : {w.status}')
print(f'render_mode : {w.render_mode}')
print(f'rows        : {w.rows_received:,}')
print(f'FPS         : {w.last_fps:.1f}')
print(f'frame_ms    : {w.avg_frame_ms:.2f}')
print(f'bytes_recv  : {w.bytes_received/1e6:.1f} MB')

status      : data-rendered
render_mode : density-only
rows        : 149,996,355
FPS         : 60.0
frame_ms    : 16.67
bytes_recv  : 1800.0 MB
